# Windhoek Traffic Congestion Analysis
**Author:** Alexander K. Moyo  
**Data source:** OpenStreetMap via OSMnx  
**Description:** This notebook walks through the full analysis pipeline —
from fetching the road network to visualising peak-hour congestion patterns.


In [ ]:
import sys
sys.path.append('../src')

import osmnx as ox
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from pipeline import (
    fetch_road_network,
    simulate_gps_observations,
    compute_congestion,
    build_map
)

ox.settings.log_console = False

## 1. Road Network

In [ ]:
edges = fetch_road_network()
print(f'Road segments: {len(edges)}')
edges[['name', 'highway', 'maxspeed', 'length']].head(10)

In [ ]:
# Road type distribution
road_counts = edges['highway'].astype(str).value_counts().head(10)
road_counts.plot(kind='barh', figsize=(8, 4), color='steelblue', edgecolor='white')
plt.title('Road segments by type — Windhoek')
plt.xlabel('Number of segments')
plt.tight_layout()
plt.savefig('../outputs/road_type_distribution.png', dpi=150)
plt.show()

## 2. Congestion Simulation

In [ ]:
edges = simulate_gps_observations(edges)
edges = compute_congestion(edges)
edges[['name', 'highway', 'speed_limit', 'speed_am', 'speed_pm', 'congestion_avg', 'severity']].head(10)

In [ ]:
# Congestion distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

edges['congestion_avg'].hist(bins=40, ax=axes[0], color='tomato', edgecolor='white')
axes[0].set_title('Distribution of Average Congestion Score')
axes[0].set_xlabel('Congestion score (0 = free flow, 1 = standstill)')

severity_order = ['Free flow', 'Light', 'Moderate', 'Heavy', 'Severe']
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']
counts = edges['severity'].value_counts().reindex(severity_order)
counts.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white', rot=30)
axes[1].set_title('Segments by Congestion Severity')
axes[1].set_ylabel('Number of segments')

plt.tight_layout()
plt.savefig('../outputs/congestion_distribution.png', dpi=150)
plt.show()

In [ ]:
# AM vs PM comparison by road type
by_type = edges.groupby('highway')[['congestion_am', 'congestion_pm']].mean().sort_values('congestion_am', ascending=False).head(8)
by_type.plot(kind='bar', figsize=(10, 4), color=['#e74c3c', '#3498db'], edgecolor='white', rot=30)
plt.title('Average Congestion by Road Type — AM vs PM Peak')
plt.ylabel('Congestion score')
plt.legend(['AM peak (07:00–09:00)', 'PM peak (16:00–18:00)'])
plt.tight_layout()
plt.savefig('../outputs/am_vs_pm_by_road_type.png', dpi=150)
plt.show()

## 3. Interactive Map

In [ ]:
map_path = build_map(edges)
from IPython.display import IFrame
IFrame(str(map_path), width='100%', height=500)